In [ ]:
from libraries import *
from parameters import *
from util import *

In [ ]:
# adata = sc.read_h5ad("/home/eraslab1/Projects/AbbasScreen/Data/NEPC_sec_screen_RNA_scanpy_raw_counts.h5ad")
# sc.pp.filter_genes(adata, min_cells=1000)
# adata_1 = sc.read_h5ad("/home/eraslab1/Projects/AbbasScreen/Data/NEPC_merged_sec_screen_RNA_scanpy_final_singlets.HVG_downstream.h5ad")
# for elem in adata_1.obs.columns:
#     adata.obs[elem] =  adata_1.obs[elem]
# adata.write("/home/eraslab1/Projects/AbbasScreen/Data/ComboScreen.h5ad")

In [ ]:
adata = sc.read_h5ad("/home/eraslab1/Projects/AbbasScreen/Data/ComboScreen.h5ad")  
# adata = adata[adata.obs[['ASCL1', 'KLF14', 'NEUROD1',                                                                                        
#          'NEUROG1', 'NR3C1', 'NTC', 'SIM1', 'TET2', 'TWIST1', 'VSX1', 'ZNF385A',                                                               
#          'ZNF547', 'ZNF660', 'ZNF776']].sum(axis=1) < 3]                                                                                       
# cols = ['ASCL1', 'KLF14', 'NEUROD1', 'NEUROG1', 'NR3C1', 'NTC', 'SIM1',                                                                      
#           'TET2', 'TWIST1', 'VSX1', 'ZNF385A', 'ZNF547', 'ZNF660', 'ZNF776']                                                                   
                                                                                                                                               
# def combine_onehot(row):                                                                                                                     
#     # Select all column names where value == 1                                                                                               
#     active = [col for col in cols if row[col] == 1]                                                                                          
#     # Join multiple actives with '+', or return 'None' if none are active                                                                    
#     return '+'.join(active) if active else 'None'                                                                                            
                                                                                                                                               
# adata.obs['perturbation'] = adata.obs[cols].apply(combine_onehot, axis=1)                                                                    
# adata.obs.to_csv("SelectedCells.csv")                                                                                                        
                                                           

In [ ]:
adata.X

In [ ]:
# Per-cell total UMIs (sum over genes). axis=1 in BOTH branches (per cell, not per gene).
gene_sums = (np.asarray(adata.X.sum(axis=1)).ravel()
             if sp.issparse(adata.X) else np.asarray(adata.X).sum(axis=1))

q1, med, q3 = np.percentile(gene_sums, [25, 50, 75])
mean = gene_sums.mean()

# Histogram with a horizontal boxplot on top (shared x): box = IQR, line = median,
# diamond = mean, whiskers = 1.5*IQR.
fig = plt.figure(figsize=(8, 4))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 6], hspace=0.05)
ax_box = fig.add_subplot(gs[0])
ax_hist = fig.add_subplot(gs[1], sharex=ax_box)

ax_hist.hist(gene_sums, bins=100, color="skyblue", edgecolor="black")
ax_hist.set_yscale("log")
ax_hist.axvline(med, color="k", ls="-", lw=1)
ax_hist.axvline(mean, color="k", ls="--", lw=1)
ax_hist.set_xlabel("Total number of UMIs per cell")
ax_hist.set_ylabel("Number of cells (log)")

bp = ax_box.boxplot(gene_sums, vert=False, widths=0.6, showfliers=False,
                    showmeans=True, patch_artist=True,
                    meanprops=dict(marker="D", markerfacecolor="k",
                                   markeredgecolor="k", markersize=4),
                    medianprops=dict(color="k", lw=1.2),
                    whiskerprops=dict(color="0.4"), capprops=dict(color="0.4"))
bp["boxes"][0].set_facecolor("skyblue")
bp["boxes"][0].set_alpha(0.6)
ax_box.set_yticks([])
for s in ax_box.spines.values():
    s.set_visible(False)
plt.setp(ax_box.get_xticklabels(), visible=False)
ax_box.set_title("Distribution of number of UMIs per cell")

ax_hist.text(0.97, 0.95,
             f"n = {len(gene_sums):,}\nmedian = {med:,.0f}\nmean = {mean:,.0f}\n"
             f"IQR = {q1:,.0f}-{q3:,.0f}",
             transform=ax_hist.transAxes, ha="right", va="top", fontsize=9,
             bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.85))

plt.show()

In [ ]:
sc.pp.normalize_total(adata, target_sum=20000)
sc.pp.log1p(adata)


In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=3000)

In [ ]:
sc.pp.scale(adata, max_value=9)

In [ ]:
sc.pp.pca(adata, n_comps=50, svd_solver='arpack')

In [ ]:
sc.pp.neighbors(adata, 
                n_neighbors=8,
                metric=par_downstream_neighbor_metric,
                n_pcs=50)

In [ ]:
sc.tl.umap(adata)

In [ ]:
adata = sc.read_h5ad("/home/eraslab1/Projects/AbbasScreen/Data/ComboScreen_processed.h5ad")  


In [ ]:
f, ax = plt.subplots(figsize=(4, 4))
sc.pl.umap(adata, color='Doxo1program_score', 
       #legend_loc='on data', 
       legend_fontoutline=3, 
       legend_fontsize=14, 
       legend_fontweight='normal', 
        color_map="coolwarm",
       ax=ax, 
       vmax=1,
       show=False, 
       size=0.3)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from statannotations.Annotator import Annotator
    _HAS_STATANNOT = True
except Exception:
    _HAS_STATANNOT = False


def compare_and_plot_violin(
    adata,
    score_key: str = "Doxo1program_score",
    group_key: str = "final_label",
    order_by: str = "mean",     # "mean" or "median"
    test: str = "wilcoxon",     # "wilcoxon" (Mann-Whitney) or "ttest"
    p_adjust: str = "fdr_bh",  # "fdr_bh" or "bonferroni" or None
    alpha: float = 0.05,
    top_pairs_to_annotate: int = 15,
    figsize=(10, 5),
):
    df = adata.obs[[score_key, group_key]].copy()
    df[score_key] = pd.to_numeric(df[score_key], errors="coerce")
    df[group_key] = df[group_key].astype(str)
    df = df.dropna(subset=[score_key, group_key])

    if df.empty:
        raise ValueError("No valid rows after dropping NA for score/group.")

    # summary + order
    summary_df = (
        df.groupby(group_key)[score_key]
        .agg(n="size", mean="mean", median="median")
        .reset_index()
    )

    if order_by not in ("mean", "median"):
        raise ValueError("order_by must be 'mean' or 'median'.")

    order = summary_df.sort_values(order_by, ascending=False)[group_key].tolist()
    df[group_key] = pd.Categorical(df[group_key], categories=order, ordered=True)

    # pairwise tests
    groups = order
    pairs, pvals = [], []
    for i, g1 in enumerate(groups):
        x1 = df.loc[df[group_key] == g1, score_key].to_numpy()
        for g2 in groups[i + 1:]:
            x2 = df.loc[df[group_key] == g2, score_key].to_numpy()

            if test.lower() in ("wilcoxon", "mannwhitney", "mw"):
                p = stats.mannwhitneyu(x1, x2, alternative="two-sided", method="asymptotic").pvalue
            elif test.lower() in ("ttest", "t-test", "t"):
                p = stats.ttest_ind(x1, x2, equal_var=False, nan_policy="omit").pvalue
            else:
                raise ValueError("test must be 'wilcoxon' (Mann-Whitney) or 'ttest'.")

            pairs.append((g1, g2))
            pvals.append(float(p))

    pairwise_df = pd.DataFrame(pairs, columns=["group1", "group2"])
    pairwise_df["pval"] = pvals

    # adjust p-values
    if p_adjust is None:
        pairwise_df["p_adj"] = pairwise_df["pval"]
        pairwise_df["p_adj_method"] = "none"
    else:
        m = len(pairwise_df)
        pa = p_adjust.lower()
        if pa == "bonferroni":
            pairwise_df["p_adj"] = np.minimum(pairwise_df["pval"] * m, 1.0)
            pairwise_df["p_adj_method"] = "bonferroni"
        elif pa in ("fdr_bh", "bh", "fdr"):
            p = pairwise_df["pval"].to_numpy()
            order_idx = np.argsort(p)
            ranked = np.empty_like(p)
            ranked[order_idx] = np.arange(1, m + 1)
            q = p * m / ranked
            q_sorted = q[order_idx]
            q_sorted = np.minimum.accumulate(q_sorted[::-1])[::-1]
            out = np.empty_like(q_sorted)
            out[order_idx] = np.minimum(q_sorted, 1.0)
            pairwise_df["p_adj"] = out
            pairwise_df["p_adj_method"] = "fdr_bh"
        else:
            raise ValueError("p_adjust must be 'fdr_bh', 'bonferroni', or None.")

    pairwise_df["significant"] = pairwise_df["p_adj"] < alpha
    pairwise_df = pairwise_df.sort_values("p_adj", ascending=True).reset_index(drop=True)

    # plot
    plt.figure(figsize=figsize)
    ax = sns.violinplot(
        data=df, x=group_key, y=score_key, order=order,
        inner="box", cut=0
    )
    ax.set_title(f"{score_key} by {group_key} (ordered by {order_by})")
    ax.tick_params(axis="x", rotation=45)

    # annotate top significant pairs
    sig_pairs = pairwise_df.loc[pairwise_df["significant"], ["group1", "group2", "p_adj"]].head(top_pairs_to_annotate)
    if not sig_pairs.empty:
        pair_list = list(map(tuple, sig_pairs[["group1", "group2"]].to_numpy()))

        if _HAS_STATANNOT:
            annot = Annotator(ax, pair_list, data=df, x=group_key, y=score_key, order=order)
            # IMPORTANT: test must be None when we supply pvalues manually
            annot.configure(test=None, text_format="star", loc="outside", comparisons_correction=None)
            annot.set_pvalues(sig_pairs["p_adj"].to_list())
            annot.annotate()
        else:
            # simple fallback stars
            def p_to_stars(p):
                return "****" if p < 1e-4 else "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else "ns"

            x_pos = {cat: i for i, cat in enumerate(order)}
            y_max = df[score_key].max()
            y_min = df[score_key].min()
            y_range = (y_max - y_min) if y_max > y_min else 1.0
            base = y_max + 0.05 * y_range
            step = 0.06 * y_range

            for k, r in enumerate(sig_pairs.itertuples(index=False)):
                x1, x2 = x_pos[r.group1], x_pos[r.group2]
                if x1 > x2:
                    x1, x2 = x2, x1
                y = base + k * step
                ax.plot([x1, x1, x2, x2], [y, y + 0.01*y_range, y + 0.01*y_range, y], linewidth=1)
                ax.text((x1 + x2)/2, y + 0.012*y_range, p_to_stars(r.p_adj),
                        ha="center", va="bottom")
            ax.set_ylim(top=base + (len(sig_pairs) + 2) * step)

    plt.tight_layout()
    plt.show()

    return summary_df, pairwise_df

In [ ]:
summary_df, pairwise_df = compare_and_plot_violin(
    adata,
    score_key="Doxo1program_score",
    group_key="final_label",
    order_by="mean",
    test="wilcoxon",
    p_adjust="fdr_bh",
    alpha=0.05,
    top_pairs_to_annotate=15,
)

In [ ]:
f, ax = plt.subplots(figsize=(4, 4))
sc.pl.umap(adata, color='phase', 
       #legend_loc='on data', 
       legend_fontoutline=3, 
       legend_fontsize=14, 
       legend_fontweight='normal', 
       ax=ax, 
       show=False, 
       size=0.3)

In [ ]:
f, ax = plt.subplots(figsize=(4, 4))
sc.pl.umap(adata, color='final_label', 
       #legend_loc='on data', 
       legend_fontoutline=3, 
       legend_fontsize=14, 
       legend_fontweight='normal', 
       ax=ax, 
       show=False, 
       size=0.3)

In [ ]:
for elem in ['phase','time_point','Doxo-1-differentiated_score', 'Neuroendocrine_score',
       'Intermediate-1_score', 'Differentiated-1_score',
       'Differentiated-2_score', 'Intermediate-3_score',
       'Intermediate-2_score', 'final_label',]:
    f, ax = plt.subplots(figsize=(4, 4))
    sc.pl.umap(adata, color=elem, 
           #legend_loc='on data', 
           legend_fontoutline=3, 
           legend_fontsize=14, 
           legend_fontweight='normal', 
           ax=ax, 
           show=False, 
           size=0.3)

In [ ]:
adata.obs["perturbation_time"] = (
    adata.obs["perturbation"].astype(str) + "_" + adata.obs["time_point"].astype(str)
)

In [ ]:
adata.obs["perturbation"].value_counts()

In [ ]:
"""
Perturbation Enrichment Analysis
=================================
Tests which perturbations are enriched in each cell state compared to NTC cells.
Uses Fisher's exact test with BH correction. Plots significant hits (FDR < 0.1).
Visualizations use log2(odds ratio) from Fisher's exact test.

Analysis 1: Based on perturbation identity only
Analysis 2: Based on perturbation × time point (day04, day10)
"""

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.lines as mlines
import seaborn as sns
import scanpy as sc
import warnings

warnings.filterwarnings("ignore")



# ---------------------------------------------------------------------------
# Core enrichment functions
# ---------------------------------------------------------------------------

def run_enrichment(adata, perturbation_col, ntc_label="NTC", cell_state_col="final_label"):
    obs = adata.obs[[perturbation_col, cell_state_col]].copy()

    ntc_cells = obs[obs[perturbation_col] == ntc_label]
    pert_cells = obs[obs[perturbation_col] != ntc_label]

    ntc_state_counts = ntc_cells[cell_state_col].value_counts()
    ntc_total = len(ntc_cells)

    perturbations = sorted(pert_cells[perturbation_col].unique())
    cell_states = sorted(obs[cell_state_col].unique())

    results = []
    for pert in perturbations:
        pert_subset = pert_cells[pert_cells[perturbation_col] == pert]
        pert_total = len(pert_subset)
        pert_state_counts = pert_subset[cell_state_col].value_counts()

        for state in cell_states:
            a = pert_state_counts.get(state, 0)
            b = pert_total - a
            c = ntc_state_counts.get(state, 0)
            d = ntc_total - c

            odds_ratio, pval = fisher_exact([[a, b], [c, d]], alternative="two-sided")

            pert_frac = a / pert_total if pert_total > 0 else 0
            ntc_frac = c / ntc_total if ntc_total > 0 else 0

            if odds_ratio == 0:
                log2_or = -LOG2OR_CAP
            elif np.isinf(odds_ratio):
                log2_or = LOG2OR_CAP
            else:
                log2_or = np.clip(np.log2(odds_ratio), -LOG2OR_CAP, LOG2OR_CAP)

            results.append({
                "perturbation": pert,
                "cell_state": state,
                "n_pert_in_state": a,
                "n_pert_total": pert_total,
                "n_ntc_in_state": c,
                "n_ntc_total": ntc_total,
                "frac_pert": round(pert_frac, 4),
                "frac_ntc": round(ntc_frac, 4),
                "odds_ratio": round(odds_ratio, 4),
                "log2_or": round(log2_or, 4),
                "pval": pval,
            })

    results_df = pd.DataFrame(results)
    if len(results_df) > 0:
        _, pval_adj, _, _ = multipletests(results_df["pval"], method="fdr_bh")
        results_df["pval_adj"] = pval_adj

    results_df = results_df.sort_values(["cell_state", "pval_adj"]).reset_index(drop=True)
    return results_df


def run_enrichment_by_timepoint(adata, ntc_label="NTC", cell_state_col="final_label"):
    obs = adata.obs[["perturbation", "perturbation_time", cell_state_col]].copy()

    obs["time"] = obs["perturbation_time"].str.extract(r"(day\d+)", expand=False)
    if obs["time"].isna().all():
        obs["time"] = obs["perturbation_time"]

    timepoints = sorted(obs["time"].dropna().unique())
    cell_states = sorted(obs[cell_state_col].unique())
    all_results = []

    for tp in timepoints:
        tp_obs = obs[obs["time"] == tp]
        ntc_cells = tp_obs[tp_obs["perturbation"] == ntc_label]
        pert_cells = tp_obs[tp_obs["perturbation"] != ntc_label]

        ntc_state_counts = ntc_cells[cell_state_col].value_counts()
        ntc_total = len(ntc_cells)

        for pert in sorted(pert_cells["perturbation"].unique()):
            pert_subset = pert_cells[pert_cells["perturbation"] == pert]
            pert_total = len(pert_subset)
            pert_state_counts = pert_subset[cell_state_col].value_counts()

            for state in cell_states:
                a = pert_state_counts.get(state, 0)
                b = pert_total - a
                c = ntc_state_counts.get(state, 0)
                d = ntc_total - c

                odds_ratio, pval = fisher_exact([[a, b], [c, d]], alternative="two-sided")

                pert_frac = a / pert_total if pert_total > 0 else 0
                ntc_frac = c / ntc_total if ntc_total > 0 else 0

                if odds_ratio == 0:
                    log2_or = -LOG2OR_CAP
                elif np.isinf(odds_ratio):
                    log2_or = LOG2OR_CAP
                else:
                    log2_or = np.clip(np.log2(odds_ratio), -LOG2OR_CAP, LOG2OR_CAP)

                all_results.append({
                    "perturbation": pert,
                    "timepoint": tp,
                    "cell_state": state,
                    "n_pert_in_state": a,
                    "n_pert_total": pert_total,
                    "n_ntc_in_state": c,
                    "n_ntc_total": ntc_total,
                    "frac_pert": round(pert_frac, 4),
                    "frac_ntc": round(ntc_frac, 4),
                    "odds_ratio": round(odds_ratio, 4),
                    "log2_or": round(log2_or, 4),
                    "pval": pval,
                })

    results_df = pd.DataFrame(all_results)
    if len(results_df) > 0:
        _, pval_adj, _, _ = multipletests(results_df["pval"], method="fdr_bh")
        results_df["pval_adj"] = pval_adj

    results_df = results_df.sort_values(["timepoint", "cell_state", "pval_adj"]).reset_index(drop=True)
    return results_df


# ---------------------------------------------------------------------------
# Plotting functions
# ---------------------------------------------------------------------------



In [ ]:
def plot_enrichment_heatmap(results_df, title, filename):
    sig = results_df[results_df["pval_adj"] < FDR_THRESHOLD].copy()
    
    sig_perts = sig["perturbation"].unique()
    print(len(sig_perts))
    sub = results_df[results_df["perturbation"].isin(sig_perts)].copy()

    #print(sub.shape)
    
    pivot_or = sub.pivot_table(index="perturbation", columns="cell_state", values="log2_or")
    print(pivot_or.shape)
    
    pivot_sig = sub.pivot_table(index="perturbation", columns="cell_state", values="pval_adj")

    mask = pivot_sig >= FDR_THRESHOLD

    annot = pivot_or.copy().astype(object)
    for r in annot.index:
        for c in annot.columns:
            try:
                p = pivot_sig.loc[r, c]
            except KeyError:
                annot.loc[r, c] = ""
                continue
            if pd.notna(p) and p < FDR_THRESHOLD:
                stars = "***" if p < 0.001 else ("**" if p < 0.01 else "*")
                annot.loc[r, c] = f"{pivot_or.loc[r, c]:.1f}{stars}"
            else:
                annot.loc[r, c] = ""

    vmax = max(abs(pivot_or.min().min()), abs(pivot_or.max().max()), 1)

    height = 6
    width = 6
    fig, ax = plt.subplots(figsize=(width, height))

    sns.heatmap(
        pivot_or,
        mask=mask,
        cmap="RdBu_r",
        center=0,
        vmin=-vmax,
        vmax=vmax,
        annot=annot,
        fmt="",
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "log2(odds ratio)", "shrink": 0.7},
        ax=ax,
    )

    for i in range(mask.shape[0]):
        for j in range(mask.shape[1]):
            if mask.iloc[i, j]:
                ax.add_patch(mpl.patches.Rectangle(
                    (j, i), 1, 1, fill=True, facecolor="#f0f0f0",
                    edgecolor="white", lw=0.5,
                ))

    ax.set_title(
        f"{title}\nlog2(OR), FDR < {FDR_THRESHOLD}  (* < 0.1,  ** < 0.01,  *** < 0.001)",
        fontsize=12, pad=12,
    )
    ax.set_xlabel("Cell state", fontsize=11)
    ax.set_ylabel("Perturbation", fontsize=11)
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {filename}")


In [ ]:
FDR_THRESHOLD = 0.05
LOG2OR_CAP = 10  # cap extreme log2(OR) for visualization

# ── Analysis 1: By perturbation ───────────────────────────────────────────
print("=" * 60)
print("Analysis 1: Enrichment by perturbation")
print("=" * 60)
results_pert = run_enrichment(
    adata,
    perturbation_col="perturbation",
    ntc_label="NTC",           # <-- adjust if your NTC label differs
    cell_state_col="final_label",
)
results_pert.to_csv("enrichment_by_perturbation.csv", index=False)
n_sig = (results_pert["pval_adj"] < FDR_THRESHOLD).sum()
print(f"  Total tests: {len(results_pert)},  Significant (FDR<{FDR_THRESHOLD}): {n_sig}")


In [ ]:

plot_enrichment_heatmap(
    results_pert,
    title="Perturbation enrichment per cell state",
    filename="enrichment_heatmap_perturbation.png",
)

In [ ]:
FDR_THRESHOLD=0.1
# ── Analysis 2: By perturbation × timepoint ──────────────────────────────
print("\n" + "=" * 60)
print("Analysis 2: Enrichment by perturbation x timepoint")
print("=" * 60)
results_time = run_enrichment_by_timepoint(
    adata,
    ntc_label="NTC",           # <-- adjust if your NTC label differs
    cell_state_col="final_label",
)
results_time.to_csv("enrichment_by_perturbation_timepoint.csv", index=False)
n_sig = (results_time["pval_adj"] < FDR_THRESHOLD).sum()
print(f"  Total tests: {len(results_time)},  Significant (FDR<{FDR_THRESHOLD}): {n_sig}")



In [ ]:
for tp in sorted(results_time["timepoint"].unique()):
    sub = results_time[results_time["timepoint"] == tp].copy()
    n_sig_tp = (sub["pval_adj"] < FDR_THRESHOLD).sum()
    print(f"\n  --- {tp}: {n_sig_tp} significant hits ---")

    plot_enrichment_heatmap(
        sub,
        title=f"Perturbation enrichment — {tp}",
        filename=f"enrichment_heatmap_{tp}.png",
    )

In [ ]:
doxo1Signature = pd.read_csv("Doxo_1_differentiated.DEGs.csv", index_col=0)

In [ ]:
sc.tl.score_genes(
        adata,
        gene_list=list(doxo1Signature.names),
        score_name="Doxo1program_score",
        use_raw=False
    )


In [ ]:
#allGenePrograms = pd.read_csv("gene_programs_total_20_onmf_20_prior_0.csv")
allGenePrograms = pd.read_csv("Programs_K14.csv", index_col=0)

In [ ]:
for elem in allGenePrograms.columns:
    geneList=list(allGenePrograms.loc[:,elem])
    
    sc.tl.score_genes(
            adata,
            gene_list=geneList,
            score_name="Program_"+elem,
            use_raw=False
        )
    f, ax = plt.subplots(figsize=(4, 4))
    sc.pl.umap(adata, color="Program_"+elem, 
           #legend_loc='on data', 
           legend_fontoutline=3, 
           legend_fontsize=14, 
           legend_fontweight='normal', 
           ax=ax,
           color_map="coolwarm",
           show=False, 
           size=0.3)


In [ ]:
for elem in ["Program_"+str(x) for x in  range(1,15,1) ]:
    f, ax = plt.subplots(figsize=(4, 4))
    sc.pl.umap(adata, color=elem, 
           #legend_loc='on data', 
           legend_fontoutline=3, 
           legend_fontsize=14, 
           legend_fontweight='normal',
           vmax=1,
           ax=ax,
           color_map="coolwarm",
           show=False, 
           size=0.3)

In [ ]:
adata.write("/home/eraslab1/Projects/AbbasScreen/Data/ComboScreen_processed.h5ad")

In [ ]:
adata.obs.to_csv("RNA_metadata.csv")

In [ ]:
adata.obs.columns

In [ ]:
ix = adata.var_names.get_loc("CDKN1A")
adata.obs["CDKN1A_expression"] = np.asarray(adata.X[:, ix]).ravel()

In [ ]:
"""
Doxo1program_score Enrichment Analysis
========================================
Tests which perturbations increase Doxo1program_score compared to NTC.
Uses Mann-Whitney U test (one-sided: perturbation > NTC) with BH correction.

For each perturbation:
  - Global test: all cells of that perturbation vs all NTC cells
  - Per cell state test: within each trajectory state separately

Analysis 1: By perturbation only
Analysis 2: By perturbation × timepoint (day04, day10)
"""

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.lines as mlines
import seaborn as sns
import scanpy as sc
import warnings

warnings.filterwarnings("ignore")

FDR_THRESHOLD = 0.3

TRAJECTORY_ORDER = [
    "Neuroendocrine",
    "Intermediate-1",
    "Intermediate-2",
    "Intermediate-3",
    "Differentiated_1",
    "Differentiated-2",
]

CELL_STATE_COL = "final_label"
NTC_LABEL = "NTC"  # <-- adjust if different


# ---------------------------------------------------------------------------
# Core test functions
# ---------------------------------------------------------------------------

def _mwu_test(pert_scores, ntc_scores):
    """One-sided Mann-Whitney U: perturbation > NTC."""
    if len(pert_scores) < 3 or len(ntc_scores) < 3:
        return np.nan, np.nan, np.nan, np.nan
    stat, pval = mannwhitneyu(pert_scores, ntc_scores, alternative="greater")
    delta_mean = pert_scores.mean() - ntc_scores.mean()
    delta_median = np.median(pert_scores) - np.median(ntc_scores)
    return stat, pval, delta_mean, delta_median


def run_score_enrichment(adata, perturbation_col, ntc_label=NTC_LABEL):
    """
    Test each perturbation vs NTC globally and per cell state.
    Returns two DataFrames: global_results, per_state_results.
    """
    obs = adata.obs[[perturbation_col, CELL_STATE_COL, SCORE_COL]].copy()

    ntc = obs[obs[perturbation_col] == ntc_label]
    perts = obs[obs[perturbation_col] != ntc_label]
    perturbation_list = sorted(perts[perturbation_col].unique())

    # --- Global test ---
    global_results = []
    for pert in perturbation_list:
        pert_scores = perts.loc[perts[perturbation_col] == pert, SCORE_COL].values
        ntc_scores = ntc[SCORE_COL].values
        stat, pval, delta_mean, delta_median = _mwu_test(pert_scores, ntc_scores)

        global_results.append({
            "perturbation": pert,
            "n_pert": len(pert_scores),
            "n_ntc": len(ntc_scores),
            "mean_pert": round(np.mean(pert_scores), 4),
            "mean_ntc": round(np.mean(ntc_scores), 4),
            "delta_mean": round(delta_mean, 4) if not np.isnan(delta_mean) else np.nan,
            "delta_median": round(delta_median, 4) if not np.isnan(delta_median) else np.nan,
            "U_stat": stat,
            "pval": pval,
        })

    global_df = pd.DataFrame(global_results)
    valid = global_df["pval"].notna()
    if valid.sum() > 0:
        _, padj, _, _ = multipletests(global_df.loc[valid, "pval"], method="fdr_bh")
        global_df.loc[valid, "pval_adj"] = padj
    else:
        global_df["pval_adj"] = np.nan
    global_df = global_df.sort_values("pval_adj").reset_index(drop=True)

    # --- Per cell state test ---
    state_results = []
    for pert in perturbation_list:
        pert_obs = perts[perts[perturbation_col] == pert]
        for state in TRAJECTORY_ORDER:
            pert_scores = pert_obs.loc[pert_obs[CELL_STATE_COL] == state, SCORE_COL].values
            ntc_scores = ntc.loc[ntc[CELL_STATE_COL] == state, SCORE_COL].values
            stat, pval, delta_mean, delta_median = _mwu_test(pert_scores, ntc_scores)

            state_results.append({
                "perturbation": pert,
                "cell_state": state,
                "n_pert": len(pert_scores),
                "n_ntc": len(ntc_scores),
                "mean_pert": round(np.mean(pert_scores), 4) if len(pert_scores) > 0 else np.nan,
                "mean_ntc": round(np.mean(ntc_scores), 4) if len(ntc_scores) > 0 else np.nan,
                "delta_mean": round(delta_mean, 4) if not np.isnan(delta_mean) else np.nan,
                "delta_median": round(delta_median, 4) if not np.isnan(delta_median) else np.nan,
                "U_stat": stat,
                "pval": pval,
            })

    state_df = pd.DataFrame(state_results)
    valid = state_df["pval"].notna()
    if valid.sum() > 0:
        _, padj, _, _ = multipletests(state_df.loc[valid, "pval"], method="fdr_bh")
        state_df.loc[valid, "pval_adj"] = padj
    else:
        state_df["pval_adj"] = np.nan
    state_df = state_df.sort_values(["cell_state", "pval_adj"]).reset_index(drop=True)

    return global_df, state_df


def run_score_enrichment_by_timepoint(adata, ntc_label=NTC_LABEL):
    """
    Test each perturbation vs NTC at the same timepoint,
    globally and per cell state.
    Returns two DataFrames: global_results, per_state_results.
    """
    obs = adata.obs[["perturbation", "perturbation_time", CELL_STATE_COL, SCORE_COL]].copy()
    obs["time"] = obs["perturbation_time"].str.extract(r"(day\d+)", expand=False)
    if obs["time"].isna().all():
        obs["time"] = obs["perturbation_time"]

    timepoints = sorted(obs["time"].dropna().unique())

    global_results = []
    state_results = []

    for tp in timepoints:
        tp_obs = obs[obs["time"] == tp]
        ntc = tp_obs[tp_obs["perturbation"] == ntc_label]
        perts = tp_obs[tp_obs["perturbation"] != ntc_label]

        for pert in sorted(perts["perturbation"].unique()):
            pert_obs = perts[perts["perturbation"] == pert]

            # Global
            pert_scores = pert_obs[SCORE_COL].values
            ntc_scores = ntc[SCORE_COL].values
            stat, pval, delta_mean, delta_median = _mwu_test(pert_scores, ntc_scores)
            global_results.append({
                "perturbation": pert,
                "timepoint": tp,
                "n_pert": len(pert_scores),
                "n_ntc": len(ntc_scores),
                "mean_pert": round(np.mean(pert_scores), 4),
                "mean_ntc": round(np.mean(ntc_scores), 4),
                "delta_mean": round(delta_mean, 4) if not np.isnan(delta_mean) else np.nan,
                "delta_median": round(delta_median, 4) if not np.isnan(delta_median) else np.nan,
                "U_stat": stat,
                "pval": pval,
            })

            # Per cell state
            for state in TRAJECTORY_ORDER:
                ps = pert_obs.loc[pert_obs[CELL_STATE_COL] == state, SCORE_COL].values
                ns = ntc.loc[ntc[CELL_STATE_COL] == state, SCORE_COL].values
                stat, pval, dm, dmed = _mwu_test(ps, ns)
                state_results.append({
                    "perturbation": pert,
                    "timepoint": tp,
                    "cell_state": state,
                    "n_pert": len(ps),
                    "n_ntc": len(ns),
                    "mean_pert": round(np.mean(ps), 4) if len(ps) > 0 else np.nan,
                    "mean_ntc": round(np.mean(ns), 4) if len(ns) > 0 else np.nan,
                    "delta_mean": round(dm, 4) if not np.isnan(dm) else np.nan,
                    "delta_median": round(dmed, 4) if not np.isnan(dmed) else np.nan,
                    "U_stat": stat,
                    "pval": pval,
                })

    global_df = pd.DataFrame(global_results)
    state_df = pd.DataFrame(state_results)

    for df in [global_df, state_df]:
        valid = df["pval"].notna()
        if valid.sum() > 0:
            _, padj, _, _ = multipletests(df.loc[valid, "pval"], method="fdr_bh")
            df.loc[valid, "pval_adj"] = padj
        else:
            df["pval_adj"] = np.nan

    global_df = global_df.sort_values(["timepoint", "pval_adj"]).reset_index(drop=True)
    state_df = state_df.sort_values(["timepoint", "cell_state", "pval_adj"]).reset_index(drop=True)

    return global_df, state_df


# ---------------------------------------------------------------------------
# Plotting functions
# ---------------------------------------------------------------------------


def plot_state_heatmap(state_df, title, filename):
    """
    Heatmap of delta_mean (pert - NTC) per perturbation × cell state.
    Only perturbations with ≥1 significant cell state shown.
    Cell states ordered along trajectory.
    """
    sig = state_df[state_df["pval_adj"] < FDR_THRESHOLD].copy()
    if sig.empty:
        print(f"  No per-state significant hits for: {title}")
        return

    sig_perts = sig["perturbation"].unique()
    sub = state_df[state_df["perturbation"].isin(sig_perts)].copy()

    # Use trajectory order for columns
    states_present = [s for s in TRAJECTORY_ORDER if s in sub["cell_state"].unique()]

    pivot_delta = sub.pivot_table(index="perturbation", columns="cell_state", values="delta_mean")
    pivot_delta = pivot_delta.reindex(columns=states_present)
    pivot_sig = sub.pivot_table(index="perturbation", columns="cell_state", values="pval_adj")
    pivot_sig = pivot_sig.reindex(columns=states_present)

    mask = (pivot_sig >= FDR_THRESHOLD) | pivot_sig.isna()

    # Annotation
    annot = pivot_delta.copy().astype(object)
    for r in annot.index:
        for c in annot.columns:
            try:
                p = pivot_sig.loc[r, c]
            except KeyError:
                annot.loc[r, c] = ""
                continue
            if pd.notna(p) and p < FDR_THRESHOLD:
                stars = "***" if p < 0.001 else ("**" if p < 0.01 else "*")
                annot.loc[r, c] = f"{pivot_delta.loc[r, c]:.2f}{stars}"
            else:
                annot.loc[r, c] = ""

    vmax = np.nanmax(np.abs(pivot_delta.values))
    if vmax == 0 or np.isnan(vmax):
        vmax = 1

    height = 10
    width = 7
    fig, ax = plt.subplots(figsize=(width, height))

    sns.heatmap(
        pivot_delta,
        mask=mask,
        cmap="Reds",
        vmin=0,
        vmax=vmax,
        annot=annot,
        fmt="",
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": f"Δ mean {SCORE_COL}", "shrink": 0.7},
        ax=ax,
    )

    for i in range(mask.shape[0]):
        for j in range(mask.shape[1]):
            if mask.iloc[i, j]:
                ax.add_patch(mpl.patches.Rectangle(
                    (j, i), 1, 1, fill=True, facecolor="#f0f0f0",
                    edgecolor="white", lw=0.5,
                ))

    ax.set_title(
        f"{title}\nΔ mean score (pert − NTC), FDR < {FDR_THRESHOLD}  (* < 0.1, ** < 0.01, *** < 0.001)",
        fontsize=11, pad=12,
    )
    ax.set_xlabel("Cell state (trajectory order →)", fontsize=11)
    ax.set_ylabel("Perturbation", fontsize=11)
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {filename}")




# ---------------------------------------------------------------------------
# Run analyses
# ---------------------------------------------------------------------------




In [ ]:
SCORE_COL = "CDKN1A_expression"

# ── Analysis 1: By perturbation ───────────────────────────────────────────
print("=" * 60)
print("Analysis 1: CDKN1A_score enrichment by perturbation")
print("=" * 60)

global_pert, state_pert = run_score_enrichment(
    adata, perturbation_col="perturbation", ntc_label=NTC_LABEL,
)


plot_state_heatmap(
    state_pert,
    title="CDKN1A_score increase vs NTC — by perturbation",
    filename="doxo1_state_heatmap_perturbation.png",
)

# ── Analysis 2: By perturbation × timepoint ──────────────────────────────
print("\n" + "=" * 60)
print("Analysis 2: CDKN1A_score enrichment by perturbation x timepoint")
print("=" * 60)

global_time, state_time = run_score_enrichment_by_timepoint(
    adata, ntc_label=NTC_LABEL,
)
global_time.to_csv("doxo1_global_by_perturbation_timepoint.csv", index=False)
state_time.to_csv("doxo1_perstate_by_perturbation_timepoint.csv", index=False)

for tp in sorted(global_time["timepoint"].unique()):
    print(f"\n  --- {tp} ---")
    sub_global = global_time[global_time["timepoint"] == tp]
    sub_state = state_time[state_time["timepoint"] == tp]

    plot_state_heatmap(
        sub_state,
        title=f"CDKN1A_score increase vs NTC — {tp}",
        filename=f"doxo1_state_heatmap_{tp}.png",
    )

In [ ]:
SCORE_COL = "Doxo1program_score"

# ── Analysis 1: By perturbation ───────────────────────────────────────────
print("=" * 60)
print("Analysis 1: Doxo1program_score enrichment by perturbation")
print("=" * 60)

global_pert, state_pert = run_score_enrichment(
    adata, perturbation_col="perturbation", ntc_label=NTC_LABEL,
)


plot_state_heatmap(
    state_pert,
    title="Doxo1program_score increase vs NTC — by perturbation",
    filename="doxo1_state_heatmap_perturbation.png",
)

# ── Analysis 2: By perturbation × timepoint ──────────────────────────────
print("\n" + "=" * 60)
print("Analysis 2: Doxo1program_score enrichment by perturbation x timepoint")
print("=" * 60)

global_time, state_time = run_score_enrichment_by_timepoint(
    adata, ntc_label=NTC_LABEL,
)
global_time.to_csv("doxo1_global_by_perturbation_timepoint.csv", index=False)
state_time.to_csv("doxo1_perstate_by_perturbation_timepoint.csv", index=False)

for tp in sorted(global_time["timepoint"].unique()):
    print(f"\n  --- {tp} ---")
    sub_global = global_time[global_time["timepoint"] == tp]
    sub_state = state_time[state_time["timepoint"] == tp]

    plot_state_heatmap(
        sub_state,
        title=f"Doxo1program_score increase vs NTC — {tp}",
        filename=f"doxo1_state_heatmap_{tp}.png",
    )

In [ ]:
# for elem in ["Program_"+str(x) for x in  range(0,20,1) ]:
#         SCORE_COL=elem

#         global_pert, state_pert = run_score_enrichment(
#             adata, perturbation_col="perturbation", ntc_label=NTC_LABEL,
#         )

#         plot_state_heatmap(
#             state_pert,
#             title= SCORE_COL +" increase vs NTC — by perturbation",
#             filename=SCORE_COL+"_heatmap_perturbation.png",
#         )

        # ── Analysis 2: By perturbation × timepoint ──────────────────────────────

#         global_time, state_time = run_score_enrichment_by_timepoint(
#             adata, ntc_label=NTC_LABEL,
#         )
       
#         for tp in sorted(global_time["timepoint"].unique()):
#             print(f"\n  --- {tp} ---")
#             sub_global = global_time[global_time["timepoint"] == tp]
#             sub_state = state_time[state_time["timepoint"] == tp]

#             plot_state_heatmap(
#                 sub_state,
#                 title=SCORE_COL+f" increase vs NTC — {tp}",
#                 filename=SCORE_COL+"f_heatmap_{tp}.png",
#             )

In [ ]:
"""
Linear Regression Analysis of KO Effects on Doxo1program_score
===============================================================
Fits OLS:  Doxo1program_score ~ gene1 + gene2 + ... + gene1:gene2 + ...

Produces three heatmaps (Differentiated_1, Differentiated-2, combined
"Differentiated state") where:
  - Rows    = all double KO pairs
  - Columns = β_gene1 | β_gene2 | β_gene1:gene2 | total effect
  - Total effect = β_gene1 + β_gene2 + β_gene1:gene2
    (the predicted shift of the double KO relative to NTC)
  - Significance tested via linear combination from the OLS model
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from itertools import combinations
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scanpy as sc
import warnings

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

SCORE_COL = "Doxo1program_score"
CELL_STATE_COL = "final_label"
NTC_LABEL = "NTC"

GENES = ["ASCL1", "NEUROG1", "NEUROD1", "TWIST1", "SIM1", "ZNF776"]

FDR_THRESHOLD = 0.1

MODEL_LABELS = {
    "Differentiated_1": ["Differentiated_1"],
    "Differentiated-2": ["Differentiated-2"],
    "Differentiated (combined)": ["Differentiated_1", "Differentiated-2"],
}


# ---------------------------------------------------------------------------
# Build design matrix
# ---------------------------------------------------------------------------

def build_design_matrix(adata):
    obs = adata.obs[["perturbation", CELL_STATE_COL, SCORE_COL]].copy()

    ntc_mask = obs["perturbation"] == NTC_LABEL

    single_ko_mask = pd.Series(False, index=obs.index)
    for gene in GENES:
        single_ko_mask |= (
            (obs["perturbation"] == gene) |
            (obs["perturbation"] == f"{gene}+NTC") |
            (obs["perturbation"] == f"NTC+{gene}")
        )

    double_ko_mask = pd.Series(False, index=obs.index)
    for ga, gb in combinations(GENES, 2):
        double_ko_mask |= (
            (obs["perturbation"] == f"{ga}+{gb}") |
            (obs["perturbation"] == f"{gb}+{ga}")
        )

    keep = ntc_mask | single_ko_mask | double_ko_mask
    df = obs[keep].copy()
    print(f"Cells in design matrix: {len(df)} "
          f"(NTC={ntc_mask.sum()}, single KO={single_ko_mask.sum()}, "
          f"double KO={double_ko_mask.sum()})")

    for gene in GENES:
        df[gene] = 0
        mask = (
            (df["perturbation"] == gene) |
            (df["perturbation"] == f"{gene}+NTC") |
            (df["perturbation"] == f"NTC+{gene}")
        )
        df.loc[mask, gene] = 1
        for other in GENES:
            if other == gene:
                continue
            mask = (
                (df["perturbation"] == f"{gene}+{other}") |
                (df["perturbation"] == f"{other}+{gene}")
            )
            df.loc[mask, gene] = 1

    main_terms = list(GENES)

    interaction_terms = []
    for ga, gb in combinations(GENES, 2):
        col = f"{ga}:{gb}"
        df[col] = df[ga] * df[gb]
        if df[col].sum() > 0:
            interaction_terms.append(col)
        else:
            df.drop(columns=[col], inplace=True)

    print(f"Main terms: {len(main_terms)}, Interaction terms: {len(interaction_terms)}")
    return df, main_terms, interaction_terms


# ---------------------------------------------------------------------------
# Fit OLS — returns both the results table AND the fitted model object
# ---------------------------------------------------------------------------

def fit_ols(df, main_terms, interaction_terms, label="model"):
    predictors = main_terms + interaction_terms
    X = df[predictors].astype(float)
    X = sm.add_constant(X)
    y = df[SCORE_COL].astype(float)

    if len(y) < len(predictors) + 2:
        print(f"  [{label}] Too few observations ({len(y)}). Skipping.")
        return None, None

    valid_cols = ["const"] + [c for c in predictors if X[c].nunique() > 1]
    X = X[valid_cols]

    model = sm.OLS(y, X).fit()

    results = []
    for term in [c for c in valid_cols if c != "const"]:
        results.append({
            "term": term,
            "term_type": "main" if term in main_terms else "interaction",
            "coef": model.params[term],
            "se": model.bse[term],
            "t_stat": model.tvalues[term],
            "pval": model.pvalues[term],
            "ci_lower": model.conf_int().loc[term, 0],
            "ci_upper": model.conf_int().loc[term, 1],
        })

    results_df = pd.DataFrame(results)
    print(f"  [{label}] n={int(model.nobs)}, R²_adj={model.rsquared_adj:.4f}, "
          f"intercept={model.params['const']:.4f}")
    return results_df, model


# ---------------------------------------------------------------------------
# Run all three models
# ---------------------------------------------------------------------------

def run_models(df_full, main_terms, interaction_terms):
    model_results = {}   # label -> results_df
    model_objects = {}   # label -> fitted OLS model

    for label, states in MODEL_LABELS.items():
        print(f"\nFitting: {label} ({states})...")
        sub = df_full[df_full[CELL_STATE_COL].isin(states)]
        if len(sub) < 10:
            print(f"  Too few cells ({len(sub)}). Skipping.")
            continue

        valid_main = [t for t in main_terms if sub[t].nunique() > 1]
        valid_inter = [t for t in interaction_terms if sub[t].sum() > 0]

        res, mod = fit_ols(sub, valid_main, valid_inter, label=label)
        if res is not None:
            model_results[label] = res
            model_objects[label] = mod

    # Joint FDR correction across all models
    all_pvals = []
    for label, res in model_results.items():
        res["model"] = label
        all_pvals.append(res)

    if len(all_pvals) > 0:
        combined = pd.concat(all_pvals, ignore_index=True)
        valid = combined["pval"].notna()
        if valid.sum() > 0:
            _, padj, _, _ = multipletests(combined.loc[valid, "pval"], method="fdr_bh")
            combined.loc[valid, "pval_adj"] = padj
        else:
            combined["pval_adj"] = np.nan

        for label in model_results:
            model_results[label] = combined[combined["model"] == label].copy()

    return model_results, model_objects


# ---------------------------------------------------------------------------
# Build pair table — now includes total effect tested via linear combination
# ---------------------------------------------------------------------------

def build_pair_table(results_df, interaction_terms, model):
    """
    For each gene pair, extract main effects, interaction, and total effect.
    Total effect = β_g1 + β_g2 + β_g1:g2, tested using t_test on the
    linear combination from the OLS model.
    """
    coef_map = results_df.set_index("term")["coef"].to_dict()
    pval_map = results_df.set_index("term")["pval_adj"].to_dict()

    # Model parameter names for building contrast vectors
    param_names = list(model.params.index)

    rows = []
    total_pvals_raw = []  # collect raw p-values for separate FDR correction

    for ga, gb in combinations(GENES, 2):
        inter_term = f"{ga}:{gb}"
        if inter_term not in interaction_terms:
            inter_term_alt = f"{gb}:{ga}"
            if inter_term_alt in interaction_terms:
                inter_term = inter_term_alt
            else:
                continue

        main_1 = coef_map.get(ga, np.nan)
        main_2 = coef_map.get(gb, np.nan)
        inter_coef = coef_map.get(inter_term, np.nan)

        # Total effect = sum of all three
        total = np.nansum([main_1, main_2, inter_coef])

        # Test total effect via linear combination: β_g1 + β_g2 + β_g1:g2 = 0
        total_pval_raw = np.nan
        try:
            contrast = np.zeros(len(param_names))
            for term in [ga, gb, inter_term]:
                if term in param_names:
                    contrast[param_names.index(term)] = 1.0
            t_test_result = model.t_test(contrast)
            total_pval_raw = float(t_test_result.pvalue)
        except Exception:
            pass

        total_pvals_raw.append(total_pval_raw)

        rows.append({
            "pair": f"{ga} + {gb}",
            "gene_1": ga,
            "gene_2": gb,
            "main_1_coef": main_1,
            "main_2_coef": main_2,
            "interaction_coef": inter_coef,
            "total_coef": round(total, 6),
            "main_1_pval": pval_map.get(ga, np.nan),
            "main_2_pval": pval_map.get(gb, np.nan),
            "interaction_pval": pval_map.get(inter_term, np.nan),
            "total_pval_raw": total_pval_raw,
        })

    pair_df = pd.DataFrame(rows)

    # FDR correct the total effect p-values
    if len(pair_df) > 0:
        valid = pair_df["total_pval_raw"].notna()
        if valid.sum() > 0:
            _, padj, _, _ = multipletests(pair_df.loc[valid, "total_pval_raw"], method="fdr_bh")
            pair_df.loc[valid, "total_pval"] = padj
        else:
            pair_df["total_pval"] = np.nan
    else:
        pair_df["total_pval"] = np.nan

    return pair_df


# ---------------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------------

def _star(p):
    if pd.isna(p) or p >= FDR_THRESHOLD:
        return ""
    return "***" if p < 0.001 else ("**" if p < 0.01 else "*")


def plot_four_column_heatmap(pair_table, model_label, filename):
    if pair_table.empty:
        print(f"  No pairs to plot for {model_label}")
        return

    n_pairs = len(pair_table)

    coefs = np.column_stack([
        pair_table["main_1_coef"].values,
        pair_table["main_2_coef"].values,
        pair_table["interaction_coef"].values,
        pair_table["total_coef"].values,
    ])
    pvals = np.column_stack([
        pair_table["main_1_pval"].values,
        pair_table["main_2_pval"].values,
        pair_table["interaction_pval"].values,
        pair_table["total_pval"].values,
    ])

    col_names = [
        "Main effect\n(Gene 1)",
        "Main effect\n(Gene 2)",
        "Interaction\n(Gene1 : Gene2)",
        "Total effect\n(Gene1 + Gene2)",
    ]
    row_names = pair_table["pair"].values

    heat_df = pd.DataFrame(coefs, index=row_names, columns=col_names)

    # Annotation: value + stars
    annot_arr = np.empty_like(coefs, dtype=object)
    for i in range(n_pairs):
        for j in range(4):
            v = coefs[i, j]
            p = pvals[i, j]
            if np.isnan(v):
                annot_arr[i, j] = ""
            else:
                s = _star(p)
                annot_arr[i, j] = f"{v:.3f}{s}"

    annot_df = pd.DataFrame(annot_arr, index=row_names, columns=col_names)

    vmax = np.nanmax(np.abs(coefs))
    if vmax == 0 or np.isnan(vmax):
        vmax = 1

    fig, ax = plt.subplots(figsize=(9, max(5, n_pairs * 0.55 + 2)))

    sns.heatmap(
        heat_df,
        cmap="RdBu_r",
        center=0,
        vmin=-vmax,
        vmax=vmax,
        annot=annot_df,
        fmt="",
        linewidths=0.8,
        linecolor="white",
        cbar_kws={"label": "β coefficient", "shrink": 0.7},
        ax=ax,
        annot_kws={"fontsize": 9},
    )

    # Add a vertical line to visually separate the total column
    ax.axvline(x=3, color="black", linewidth=2)

    ax.set_ylabel("Gene pair (double KO)", fontsize=11)
    ax.set_xlabel("")
    ax.set_title(
        f"{model_label}\n"
        f"Effect of KOs on {SCORE_COL}\n"
        f"Total = β_g1 + β_g2 + β_g1:g2  (predicted DKO shift from NTC)\n"
        f"(* FDR<0.1, **<0.01, ***<0.001)",
        fontsize=10, pad=14,
    )
    plt.xticks(rotation=0, ha="center", fontsize=10)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {filename}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------


print("=" * 70)
print(f"Linear regression: {SCORE_COL} ~ Σ gene_i + Σ gene_i:gene_j")
print(f"Genes: {GENES}")
print("=" * 70)

df, main_terms, interaction_terms = build_design_matrix(adata)

print("\nCell counts:")
for gene in GENES:
    mask = (df[gene] == 1) & (df[main_terms].drop(columns=[gene]).sum(axis=1) == 0)
    print(f"  {gene} single KO: {mask.sum()}")
for ga, gb in combinations(GENES, 2):
    col = f"{ga}:{gb}"
    if col in interaction_terms:
        print(f"  {ga}+{gb} double KO: {(df[col] == 1).sum()}")
ntc_n = (df[main_terms].sum(axis=1) == 0).sum()
print(f"  NTC: {ntc_n}")

model_results, model_objects = run_models(df, main_terms, interaction_terms)

all_coefs = pd.concat(model_results.values(), ignore_index=True)
all_coefs.to_csv("regression_coefficients.csv", index=False)
print("\nCoefficients saved to regression_coefficients.csv")

for label, res in model_results.items():
    print(f"\n{'=' * 70}")
    print(f"  {label}")
    print(f"{'=' * 70}")

    sig_main = res[(res["term_type"] == "main") & (res["pval_adj"] < FDR_THRESHOLD)]
    print(f"\n  Significant main effects:")
    if len(sig_main) > 0:
        for _, row in sig_main.sort_values("pval_adj").iterrows():
            d = "↑" if row["coef"] > 0 else "↓"
            print(f"    {row['term']:10s}  β={row['coef']:+.4f} (SE={row['se']:.4f})  "
                  f"FDR={row['pval_adj']:.2e}  {d}")
    else:
        print("    None")

    sig_inter = res[(res["term_type"] == "interaction") & (res["pval_adj"] < FDR_THRESHOLD)]
    print(f"\n  Significant interactions:")
    if len(sig_inter) > 0:
        for _, row in sig_inter.sort_values("pval_adj").iterrows():
            interp = "SYNERGISTIC" if row["coef"] > 0 else "ANTAGONISTIC"
            print(f"    {row['term']:20s}  β={row['coef']:+.4f} (SE={row['se']:.4f})  "
                  f"FDR={row['pval_adj']:.2e}  [{interp}]")
    else:
        print("    None")

print("\n" + "=" * 70)
print("Generating heatmaps...")
print("=" * 70)

for label, res in model_results.items():
    mod = model_objects[label]
    pair_table = build_pair_table(res, interaction_terms, mod)
    safe_name = label.replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "")
    plot_four_column_heatmap(
        pair_table,
        model_label=label,
        filename=f"heatmap_{safe_name}.png",
    )
    pair_table.to_csv(f"pair_effects_{safe_name}.csv", index=False)
    print(f"  Pair table saved: pair_effects_{safe_name}.csv")

In [ ]:
"""
Linear Regression Analysis of KO Effects on Doxo1program_score
===============================================================
Fits OLS:  Doxo1program_score ~ gene1 + gene2 + ... + gene1:gene2 + ...

Produces three heatmaps (Differentiated_1, Differentiated-2, combined
"Differentiated state") where:
  - Rows    = all double KO pairs
  - Columns = β_gene1 | β_gene2 | β_gene1:gene2 | total effect
  - Total effect = β_gene1 + β_gene2 + β_gene1:gene2
    (the predicted shift of the double KO relative to NTC)
  - Significance tested via linear combination from the OLS model
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from itertools import combinations
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scanpy as sc
import warnings

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

SCORE_COL = "CDKN1A_expression"
CELL_STATE_COL = "final_label"
NTC_LABEL = "NTC"

GENES = ["ASCL1", "NEUROG1", "NEUROD1", "TWIST1", "SIM1", "ZNF776", "KLF14"]

FDR_THRESHOLD = 0.1

MODEL_LABELS = {
    "Differentiated_1": ["Differentiated_1"],
    "Differentiated-2": ["Differentiated-2"],
    "Differentiated (combined)": ["Differentiated_1", "Differentiated-2"],
}


# ---------------------------------------------------------------------------
# Build design matrix
# ---------------------------------------------------------------------------

def build_design_matrix(adata):
    obs = adata.obs[["perturbation", CELL_STATE_COL, SCORE_COL]].copy()

    ntc_mask = obs["perturbation"] == NTC_LABEL

    single_ko_mask = pd.Series(False, index=obs.index)
    for gene in GENES:
        single_ko_mask |= (
            (obs["perturbation"] == gene) |
            (obs["perturbation"] == f"{gene}+NTC") |
            (obs["perturbation"] == f"NTC+{gene}")
        )

    double_ko_mask = pd.Series(False, index=obs.index)
    for ga, gb in combinations(GENES, 2):
        double_ko_mask |= (
            (obs["perturbation"] == f"{ga}+{gb}") |
            (obs["perturbation"] == f"{gb}+{ga}")
        )

    keep = ntc_mask | single_ko_mask | double_ko_mask
    df = obs[keep].copy()
    print(f"Cells in design matrix: {len(df)} "
          f"(NTC={ntc_mask.sum()}, single KO={single_ko_mask.sum()}, "
          f"double KO={double_ko_mask.sum()})")

    for gene in GENES:
        df[gene] = 0
        mask = (
            (df["perturbation"] == gene) |
            (df["perturbation"] == f"{gene}+NTC") |
            (df["perturbation"] == f"NTC+{gene}")
        )
        df.loc[mask, gene] = 1
        for other in GENES:
            if other == gene:
                continue
            mask = (
                (df["perturbation"] == f"{gene}+{other}") |
                (df["perturbation"] == f"{other}+{gene}")
            )
            df.loc[mask, gene] = 1

    main_terms = list(GENES)

    interaction_terms = []
    for ga, gb in combinations(GENES, 2):
        col = f"{ga}:{gb}"
        df[col] = df[ga] * df[gb]
        if df[col].sum() > 0:
            interaction_terms.append(col)
        else:
            df.drop(columns=[col], inplace=True)

    print(f"Main terms: {len(main_terms)}, Interaction terms: {len(interaction_terms)}")
    return df, main_terms, interaction_terms


# ---------------------------------------------------------------------------
# Fit OLS — returns both the results table AND the fitted model object
# ---------------------------------------------------------------------------

def fit_ols(df, main_terms, interaction_terms, label="model"):
    predictors = main_terms + interaction_terms
    X = df[predictors].astype(float)
    X = sm.add_constant(X)
    y = df[SCORE_COL].astype(float)

    if len(y) < len(predictors) + 2:
        print(f"  [{label}] Too few observations ({len(y)}). Skipping.")
        return None, None

    valid_cols = ["const"] + [c for c in predictors if X[c].nunique() > 1]
    X = X[valid_cols]

    model = sm.OLS(y, X).fit()

    results = []
    for term in [c for c in valid_cols if c != "const"]:
        results.append({
            "term": term,
            "term_type": "main" if term in main_terms else "interaction",
            "coef": model.params[term],
            "se": model.bse[term],
            "t_stat": model.tvalues[term],
            "pval": model.pvalues[term],
            "ci_lower": model.conf_int().loc[term, 0],
            "ci_upper": model.conf_int().loc[term, 1],
        })

    results_df = pd.DataFrame(results)
    print(f"  [{label}] n={int(model.nobs)}, R²_adj={model.rsquared_adj:.4f}, "
          f"intercept={model.params['const']:.4f}")
    return results_df, model


# ---------------------------------------------------------------------------
# Run all three models
# ---------------------------------------------------------------------------

def run_models(df_full, main_terms, interaction_terms):
    model_results = {}   # label -> results_df
    model_objects = {}   # label -> fitted OLS model

    for label, states in MODEL_LABELS.items():
        print(f"\nFitting: {label} ({states})...")
        sub = df_full[df_full[CELL_STATE_COL].isin(states)]
        if len(sub) < 10:
            print(f"  Too few cells ({len(sub)}). Skipping.")
            continue

        valid_main = [t for t in main_terms if sub[t].nunique() > 1]
        valid_inter = [t for t in interaction_terms if sub[t].sum() > 0]

        res, mod = fit_ols(sub, valid_main, valid_inter, label=label)
        if res is not None:
            model_results[label] = res
            model_objects[label] = mod

    # Joint FDR correction across all models
    all_pvals = []
    for label, res in model_results.items():
        res["model"] = label
        all_pvals.append(res)

    if len(all_pvals) > 0:
        combined = pd.concat(all_pvals, ignore_index=True)
        valid = combined["pval"].notna()
        if valid.sum() > 0:
            _, padj, _, _ = multipletests(combined.loc[valid, "pval"], method="fdr_bh")
            combined.loc[valid, "pval_adj"] = padj
        else:
            combined["pval_adj"] = np.nan

        for label in model_results:
            model_results[label] = combined[combined["model"] == label].copy()

    return model_results, model_objects


# ---------------------------------------------------------------------------
# Build pair table — now includes total effect tested via linear combination
# ---------------------------------------------------------------------------

def build_pair_table(results_df, interaction_terms, model):
    """
    For each gene pair, extract main effects, interaction, and total effect.
    Total effect = β_g1 + β_g2 + β_g1:g2, tested using t_test on the
    linear combination from the OLS model.
    """
    coef_map = results_df.set_index("term")["coef"].to_dict()
    pval_map = results_df.set_index("term")["pval_adj"].to_dict()

    # Model parameter names for building contrast vectors
    param_names = list(model.params.index)

    rows = []
    total_pvals_raw = []  # collect raw p-values for separate FDR correction

    for ga, gb in combinations(GENES, 2):
        inter_term = f"{ga}:{gb}"
        if inter_term not in interaction_terms:
            inter_term_alt = f"{gb}:{ga}"
            if inter_term_alt in interaction_terms:
                inter_term = inter_term_alt
            else:
                continue

        main_1 = coef_map.get(ga, np.nan)
        main_2 = coef_map.get(gb, np.nan)
        inter_coef = coef_map.get(inter_term, np.nan)

        # Total effect = sum of all three
        total = np.nansum([main_1, main_2, inter_coef])

        # Test total effect via linear combination: β_g1 + β_g2 + β_g1:g2 = 0
        total_pval_raw = np.nan
        try:
            contrast = np.zeros(len(param_names))
            for term in [ga, gb, inter_term]:
                if term in param_names:
                    contrast[param_names.index(term)] = 1.0
            t_test_result = model.t_test(contrast)
            total_pval_raw = float(t_test_result.pvalue)
        except Exception:
            pass

        total_pvals_raw.append(total_pval_raw)

        rows.append({
            "pair": f"{ga} + {gb}",
            "gene_1": ga,
            "gene_2": gb,
            "main_1_coef": main_1,
            "main_2_coef": main_2,
            "interaction_coef": inter_coef,
            "total_coef": round(total, 6),
            "main_1_pval": pval_map.get(ga, np.nan),
            "main_2_pval": pval_map.get(gb, np.nan),
            "interaction_pval": pval_map.get(inter_term, np.nan),
            "total_pval_raw": total_pval_raw,
        })

    pair_df = pd.DataFrame(rows)

    # FDR correct the total effect p-values
    if len(pair_df) > 0:
        valid = pair_df["total_pval_raw"].notna()
        if valid.sum() > 0:
            _, padj, _, _ = multipletests(pair_df.loc[valid, "total_pval_raw"], method="fdr_bh")
            pair_df.loc[valid, "total_pval"] = padj
        else:
            pair_df["total_pval"] = np.nan
    else:
        pair_df["total_pval"] = np.nan

    return pair_df


# ---------------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------------

def _star(p):
    if pd.isna(p) or p >= FDR_THRESHOLD:
        return ""
    return "***" if p < 0.001 else ("**" if p < 0.01 else "*")


def plot_four_column_heatmap(pair_table, model_label, filename):
    if pair_table.empty:
        print(f"  No pairs to plot for {model_label}")
        return

    n_pairs = len(pair_table)

    coefs = np.column_stack([
        pair_table["main_1_coef"].values,
        pair_table["main_2_coef"].values,
        pair_table["interaction_coef"].values,
        pair_table["total_coef"].values,
    ])
    pvals = np.column_stack([
        pair_table["main_1_pval"].values,
        pair_table["main_2_pval"].values,
        pair_table["interaction_pval"].values,
        pair_table["total_pval"].values,
    ])

    col_names = [
        "Main effect\n(Gene 1)",
        "Main effect\n(Gene 2)",
        "Interaction\n(Gene1 : Gene2)",
        "Total effect\n(Gene1 + Gene2)",
    ]
    row_names = pair_table["pair"].values

    heat_df = pd.DataFrame(coefs, index=row_names, columns=col_names)

    # Annotation: value + stars
    annot_arr = np.empty_like(coefs, dtype=object)
    for i in range(n_pairs):
        for j in range(4):
            v = coefs[i, j]
            p = pvals[i, j]
            if np.isnan(v):
                annot_arr[i, j] = ""
            else:
                s = _star(p)
                annot_arr[i, j] = f"{v:.3f}{s}"

    annot_df = pd.DataFrame(annot_arr, index=row_names, columns=col_names)

    vmax = np.nanmax(np.abs(coefs))
    if vmax == 0 or np.isnan(vmax):
        vmax = 1

    fig, ax = plt.subplots(figsize=(9, max(5, n_pairs * 0.55 + 2)))

    sns.heatmap(
        heat_df,
        cmap="RdBu_r",
        center=0,
        vmin=-vmax,
        vmax=vmax,
        annot=annot_df,
        fmt="",
        linewidths=0.8,
        linecolor="white",
        cbar_kws={"label": "β coefficient", "shrink": 0.7},
        ax=ax,
        annot_kws={"fontsize": 9},
    )

    # Add a vertical line to visually separate the total column
    ax.axvline(x=3, color="black", linewidth=2)

    ax.set_ylabel("Gene pair (double KO)", fontsize=11)
    ax.set_xlabel("")
    ax.set_title(
        f"{model_label}\n"
        f"Effect of KOs on {SCORE_COL}\n"
        f"Total = β_g1 + β_g2 + β_g1:g2  (predicted DKO shift from NTC)\n"
        f"(* FDR<0.1, **<0.01, ***<0.001)",
        fontsize=10, pad=14,
    )
    plt.xticks(rotation=0, ha="center", fontsize=10)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {filename}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------


print("=" * 70)
print(f"Linear regression: {SCORE_COL} ~ Σ gene_i + Σ gene_i:gene_j")
print(f"Genes: {GENES}")
print("=" * 70)

df, main_terms, interaction_terms = build_design_matrix(adata)

print("\nCell counts:")
for gene in GENES:
    mask = (df[gene] == 1) & (df[main_terms].drop(columns=[gene]).sum(axis=1) == 0)
    print(f"  {gene} single KO: {mask.sum()}")
for ga, gb in combinations(GENES, 2):
    col = f"{ga}:{gb}"
    if col in interaction_terms:
        print(f"  {ga}+{gb} double KO: {(df[col] == 1).sum()}")
ntc_n = (df[main_terms].sum(axis=1) == 0).sum()
print(f"  NTC: {ntc_n}")

model_results, model_objects = run_models(df, main_terms, interaction_terms)

all_coefs = pd.concat(model_results.values(), ignore_index=True)
all_coefs.to_csv("regression_coefficients.csv", index=False)
print("\nCoefficients saved to regression_coefficients.csv")

for label, res in model_results.items():
    print(f"\n{'=' * 70}")
    print(f"  {label}")
    print(f"{'=' * 70}")

    sig_main = res[(res["term_type"] == "main") & (res["pval_adj"] < FDR_THRESHOLD)]
    print(f"\n  Significant main effects:")
    if len(sig_main) > 0:
        for _, row in sig_main.sort_values("pval_adj").iterrows():
            d = "↑" if row["coef"] > 0 else "↓"
            print(f"    {row['term']:10s}  β={row['coef']:+.4f} (SE={row['se']:.4f})  "
                  f"FDR={row['pval_adj']:.2e}  {d}")
    else:
        print("    None")

    sig_inter = res[(res["term_type"] == "interaction") & (res["pval_adj"] < FDR_THRESHOLD)]
    print(f"\n  Significant interactions:")
    if len(sig_inter) > 0:
        for _, row in sig_inter.sort_values("pval_adj").iterrows():
            interp = "SYNERGISTIC" if row["coef"] > 0 else "ANTAGONISTIC"
            print(f"    {row['term']:20s}  β={row['coef']:+.4f} (SE={row['se']:.4f})  "
                  f"FDR={row['pval_adj']:.2e}  [{interp}]")
    else:
        print("    None")

print("\n" + "=" * 70)
print("Generating heatmaps...")
print("=" * 70)

for label, res in model_results.items():
    mod = model_objects[label]
    pair_table = build_pair_table(res, interaction_terms, mod)
    safe_name = label.replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "")
    plot_four_column_heatmap(
        pair_table,
        model_label=label,
        filename=f"heatmap_{safe_name}.png",
    )
    pair_table.to_csv(f"pair_effects_{safe_name}.csv", index=False)
    print(f"  Pair table saved: pair_effects_{safe_name}.csv")

In [ ]:
"""
Pseudotime Density Analysis of Perturbations
=============================================
1. Computes diffusion pseudotime along the differentiation trajectory
2. Tests which perturbations shift significantly toward intermediate/differentiated states
3. Plots density distributions ONLY for significantly shifted perturbations

Assumes:
  - adata has a precomputed embedding (e.g. UMAP) and neighbors graph
  - adata.obs["perturbation"] contains perturbation labels
  - adata.obs["final_label"] contains cell state annotations
  - Trajectory goes: Neuroendocrine → Intermediate → Differentiated
"""

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, gaussian_kde
from statsmodels.stats.multitest import multipletests
import warnings

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

SCORE_COL = "Doxo1program_score"
CELL_STATE_COL = "final_label"
NTC_LABEL = "NTC"

TRAJECTORY_ORDER = [
    "Neuroendocrine",
    "Intermediate-1",
    "Intermediate-2",
    "Intermediate-3",
    "Differentiated_1",
    "Differentiated-2",
]

# Perturbations to test (None = auto-detect all with >= 20 cells)
PERTURBATIONS_OF_INTEREST = None

FDR_THRESHOLD = 0.1


# ---------------------------------------------------------------------------
# Step 1: Compute diffusion pseudotime
# ---------------------------------------------------------------------------

def compute_pseudotime(adata, root_state="Neuroendocrine", recompute_neighbors=False):
    if "neighbors" not in adata.uns or recompute_neighbors:
        print("Computing neighbors...")
        sc.pp.neighbors(adata, n_neighbors=30, use_rep="X_pca")

    if "X_diffmap" not in adata.obsm or recompute_neighbors:
        print("Computing diffusion map...")
        sc.tl.diffmap(adata, n_comps=15)

    root_mask = adata.obs[CELL_STATE_COL] == root_state
    if root_mask.sum() == 0:
        for state in TRAJECTORY_ORDER:
            root_mask = adata.obs[CELL_STATE_COL] == state
            if root_mask.sum() > 0:
                root_state = state
                print(f"  Root state not found, using '{state}'")
                break

    diffmap = adata.obsm["X_diffmap"]
    root_cells = diffmap[root_mask.values]
    centroid = root_cells.mean(axis=0)
    distances = np.linalg.norm(root_cells - centroid, axis=1)
    root_idx = np.where(root_mask.values)[0][np.argmin(distances)]

    adata.uns["iroot"] = root_idx
    print(f"Root cell index: {root_idx} (state: {root_state})")

    print("Computing diffusion pseudotime...")
    sc.tl.dpt(adata, n_dcs=10)

    inf_mask = np.isinf(adata.obs["dpt_pseudotime"].values)
    if inf_mask.sum() > 0:
        print(f"  Replacing {inf_mask.sum()} infinite pseudotime values with max")
        max_pt = adata.obs.loc[~inf_mask, "dpt_pseudotime"].max()
        adata.obs.loc[inf_mask, "dpt_pseudotime"] = max_pt

    # Validate direction
    median_pt = adata.obs.groupby(CELL_STATE_COL)["dpt_pseudotime"].median()
    present_states = [s for s in TRAJECTORY_ORDER if s in median_pt.index]
    medians = median_pt[present_states]
    print("\nMedian pseudotime per state:")
    for state in present_states:
        print(f"  {state:20s}: {medians[state]:.4f}")

    if len(medians) >= 2 and medians.iloc[0] > medians.iloc[-1]:
        print("\n  Pseudotime reversed — flipping...")
        adata.obs["dpt_pseudotime"] = (
            adata.obs["dpt_pseudotime"].max() - adata.obs["dpt_pseudotime"]
        )

    return adata


# ---------------------------------------------------------------------------
# Step 2: Density estimation
# ---------------------------------------------------------------------------

def compute_densities(adata, perturbations, n_points=500):
    pt = adata.obs["dpt_pseudotime"]
    x_grid = np.linspace(pt.min(), pt.max(), n_points)

    densities = {}
    for pert in perturbations:
        mask = adata.obs["perturbation"] == pert
        vals = pt[mask].values
        if len(vals) < 5:
            continue
        try:
            kde = gaussian_kde(vals, bw_method="scott")
            densities[pert] = kde(x_grid)
        except Exception:
            continue

    return x_grid, densities


# ---------------------------------------------------------------------------
# Step 3: KS test + filter for significant shift toward differentiation
# ---------------------------------------------------------------------------

def test_pseudotime_shifts(adata, perturbations):
    pt = adata.obs["dpt_pseudotime"]
    ntc_pt = pt[adata.obs["perturbation"] == NTC_LABEL].values

    results = []
    for pert in perturbations:
        if pert == NTC_LABEL:
            continue
        pert_pt = pt[adata.obs["perturbation"] == pert].values
        if len(pert_pt) < 5:
            continue

        ks_stat, ks_pval = ks_2samp(pert_pt, ntc_pt)
        delta_median = np.median(pert_pt) - np.median(ntc_pt)
        delta_mean = np.mean(pert_pt) - np.mean(ntc_pt)

        results.append({
            "perturbation": pert,
            "n_cells": len(pert_pt),
            "median_pt": round(np.median(pert_pt), 4),
            "median_ntc": round(np.median(ntc_pt), 4),
            "delta_median": round(delta_median, 4),
            "delta_mean": round(delta_mean, 4),
            "ks_stat": round(ks_stat, 4),
            "ks_pval": ks_pval,
        })

    df = pd.DataFrame(results)
    if len(df) > 0:
        _, padj, _, _ = multipletests(df["ks_pval"], method="fdr_bh")
        df["ks_pval_adj"] = padj
    df = df.sort_values("ks_pval_adj").reset_index(drop=True)
    return df


def filter_significant_differentiation_shift(stats_df):
    """
    Keep only perturbations that:
      1. Have a significant KS test (FDR < threshold)
      2. Shift TOWARD intermediate/differentiated states (positive delta_median,
         since pseudotime increases along Neuro → Intermediate → Differentiated)
    """
    sig = stats_df[
        (stats_df["ks_pval_adj"] < FDR_THRESHOLD) &
        (stats_df["delta_median"] > 0)
    ].copy()
    return sig


# ---------------------------------------------------------------------------
# Step 4: Plotting (only significant perturbations)
# ---------------------------------------------------------------------------

def _add_state_boundaries(ax, adata):
    pt = adata.obs["dpt_pseudotime"]
    states = [s for s in TRAJECTORY_ORDER if s in adata.obs[CELL_STATE_COL].unique()]

    medians = {}
    for state in states:
        vals = pt[adata.obs[CELL_STATE_COL] == state]
        if len(vals) > 0:
            medians[state] = np.median(vals)

    ymax = ax.get_ylim()[1]
    for state, med in medians.items():
        ax.axvline(med, color="#cccccc", linestyle=":", linewidth=0.8, alpha=0.7)
        ax.text(med, ymax * 0.95, state, rotation=90, fontsize=6,
                va="top", ha="right", alpha=0.6)


def plot_pseudotime_umap(adata, filename="pseudotime_umap.png"):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sc.pl.umap(adata, color="dpt_pseudotime", ax=axes[0], show=False,
               title="Diffusion pseudotime", color_map="viridis", frameon=False)
    sc.pl.umap(adata, color=CELL_STATE_COL, ax=axes[1], show=False,
               title="Cell states", frameon=False)
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {filename}")


def plot_density_overlay(adata, x_grid, densities, sig_perts, stats_df,
                         filename="pseudotime_density_sig_overlay.png"):
    """All significant perturbations overlaid on NTC."""
    fig, ax = plt.subplots(figsize=(12, 5))

    if NTC_LABEL in densities:
        ax.fill_between(x_grid, densities[NTC_LABEL], alpha=0.3, color="#bdbdbd", label="NTC")
        ax.plot(x_grid, densities[NTC_LABEL], color="#757575", linewidth=2)

    cmap = plt.cm.get_cmap("tab20", len(sig_perts))
    for i, pert in enumerate(sig_perts):
        if pert not in densities:
            continue
        row = stats_df[stats_df["perturbation"] == pert].iloc[0]
        label = f"{pert} (Δmed={row['delta_median']:+.3f}, FDR={row['ks_pval_adj']:.1e})"
        ax.plot(x_grid, densities[pert], color=cmap(i), linewidth=1.5,
                alpha=0.85, label=label)

    _add_state_boundaries(ax, adata)

    ax.set_xlabel("Pseudotime", fontsize=12)
    ax.set_ylabel("Density", fontsize=12)
    ax.set_title(
        "Perturbations with significant shift toward differentiation\n"
        f"(KS test FDR<{FDR_THRESHOLD}, positive Δ median pseudotime)",
        fontsize=12,
    )
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, frameon=True)
    ax.set_xlim(x_grid[0], x_grid[-1])
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {filename}")


def plot_density_ridge(adata, x_grid, densities, sig_perts, stats_df,
                       filename="pseudotime_density_sig_ridge.png"):
    """Ridge plot: NTC at bottom, significant perturbations stacked."""
    perts_to_plot = [NTC_LABEL] + list(sig_perts)
    perts_to_plot = [p for p in perts_to_plot if p in densities]
    n_perts = len(perts_to_plot)

    fig, axes = plt.subplots(n_perts, 1, figsize=(10, n_perts * 0.7 + 2),
                             sharex=True)
    if n_perts == 1:
        axes = [axes]

    cmap = plt.cm.get_cmap("tab20", n_perts)

    for i, pert in enumerate(perts_to_plot):
        ax = axes[i]
        density = densities.get(pert, np.zeros_like(x_grid))

        if pert == NTC_LABEL:
            color = "#bdbdbd"
            label = "NTC"
        else:
            color = cmap(i)
            row = stats_df[stats_df["perturbation"] == pert]
            if len(row) > 0:
                r = row.iloc[0]
                label = f"{pert}  (Δmed={r['delta_median']:+.3f})"
            else:
                label = pert

        ax.fill_between(x_grid, density, alpha=0.6, color=color)
        ax.plot(x_grid, density, color="black", linewidth=0.5)

        ax.set_ylabel("")
        ax.set_yticks([])
        ax.text(-0.01, 0.5, label, transform=ax.transAxes, fontsize=8,
                va="center", ha="right")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_visible(False)
        if i < n_perts - 1:
            ax.spines["bottom"].set_visible(False)
            ax.tick_params(axis="x", length=0)

    axes[-1].set_xlabel("Pseudotime", fontsize=11)
    fig.suptitle(
        "Perturbations with significant shift toward differentiation\n"
        f"(KS test FDR<{FDR_THRESHOLD}, positive Δ median)",
        fontsize=12, y=1.01,
    )
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {filename}")


def plot_density_faceted(adata, x_grid, densities, sig_perts, stats_df,
                         filename="pseudotime_density_sig_faceted.png"):
    """Each significant perturbation in its own panel, overlaid on NTC."""
    perts_to_plot = [p for p in sig_perts if p in densities]
    n_perts = len(perts_to_plot)
    if n_perts == 0:
        print("  No perturbations to plot in faceted view")
        return

    n_cols = min(4, n_perts)
    n_rows = int(np.ceil(n_perts / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows),
                             sharex=True, sharey=True, squeeze=False)

    ntc_density = densities.get(NTC_LABEL, np.zeros_like(x_grid))
    cmap = plt.cm.get_cmap("tab20", n_perts)

    for idx, pert in enumerate(perts_to_plot):
        ax = axes[idx // n_cols, idx % n_cols]

        ax.fill_between(x_grid, ntc_density, alpha=0.25, color="#bdbdbd")
        ax.plot(x_grid, ntc_density, color="#999999", linewidth=1, linestyle="--",
                label="NTC")

        color = cmap(idx)
        ax.fill_between(x_grid, densities[pert], alpha=0.5, color=color)
        ax.plot(x_grid, densities[pert], color=color, linewidth=1.5, label=pert)

        row = stats_df[stats_df["perturbation"] == pert]
        if len(row) > 0:
            r = row.iloc[0]
            title = (f"{pert}\n"
                     f"Δmed={r['delta_median']:+.3f}, "
                     f"KS={r['ks_stat']:.3f}, "
                     f"FDR={r['ks_pval_adj']:.1e}")
        else:
            title = pert
        ax.set_title(title, fontsize=8, pad=4)
        ax.set_xlim(x_grid[0], x_grid[-1])
        ax.legend(fontsize=6, loc="upper right")

        if idx // n_cols == n_rows - 1:
            ax.set_xlabel("Pseudotime", fontsize=9)
        if idx % n_cols == 0:
            ax.set_ylabel("Density", fontsize=9)

    for idx in range(n_perts, n_rows * n_cols):
        axes[idx // n_cols, idx % n_cols].set_visible(False)

    fig.suptitle(
        "Perturbations shifted toward intermediate/differentiated states\n"
        f"(KS test FDR<{FDR_THRESHOLD}, positive Δ median; grey dashed = NTC)",
        fontsize=12, y=1.02,
    )
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {filename}")


def plot_median_shift_barplot(stats_df, sig_perts,
                              filename="pseudotime_median_shift_sig.png"):
    """Bar plot of median pseudotime shift for significant perturbations only."""
    df = stats_df[stats_df["perturbation"].isin(sig_perts)].copy()
    if df.empty:
        print("  No significant perturbations for bar plot")
        return

    df = df.sort_values("delta_median")

    fig, ax = plt.subplots(figsize=(8, max(4, len(df) * 0.4 + 1)))
    colors = plt.cm.Reds(np.linspace(0.3, 1.0, len(df)))

    ax.barh(df["perturbation"], df["delta_median"], color=colors,
            edgecolor="black", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xlabel("Δ median pseudotime (perturbation − NTC)", fontsize=11)
    ax.set_ylabel("Perturbation", fontsize=11)
    ax.set_title(
        "Pseudotime shift toward differentiation\n"
        f"(significant perturbations only, KS FDR<{FDR_THRESHOLD})",
        fontsize=12,
    )

    for i, (_, row) in enumerate(df.iterrows()):
        stars = "***" if row["ks_pval_adj"] < 0.001 else (
            "**" if row["ks_pval_adj"] < 0.01 else "*")
        ax.text(row["delta_median"] + 0.003, i, stars, va="center",
                ha="left", fontsize=9, fontweight="bold")

    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {filename}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------




In [ ]:
adata.obs